# Dhivehi CSM-1B — five-speaker audio test
Runs on a Google Colab **GPU**, not offline on Android. This is to test checkpoint access and voice quality. The model needs several gigabytes of GPU memory. No paid API required, but Colab free GPU availability is not guaranteed.
Before running: Runtime > Change runtime type > T4 GPU. Accept the model conditions at https://huggingface.co/alakxender/csm-1b-dhivehi-5-spk-gd .


In [ ]:
!pip -q install -U 'transformers>=4.52.1,<5' accelerate huggingface_hub soundfile


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # Read-only Hugging Face token; do not paste tokens into shared notebooks.


In [ ]:
import torch
from transformers import CsmForConditionalGeneration, AutoProcessor
MODEL = 'alakxender/csm-1b-dhivehi-5-spk-gd'
assert torch.cuda.is_available(), 'GPU not available. In Colab select Runtime > Change runtime type > T4 GPU.'
processor = AutoProcessor.from_pretrained(MODEL)
model = CsmForConditionalGeneration.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto')
print('Model loaded. GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Edit text and speaker number 0..4, then run this cell to generate WAV.
speaker_id = 0
dhivehi_text = 'އަހަރެން މިއަދު ވަރަށް އުފާވެއްޖެ.'
assert speaker_id in range(5)
conversation = [{'role': str(speaker_id), 'content': [{'type': 'text', 'text': dhivehi_text}]}]
inputs = processor.apply_chat_template(conversation, tokenize=True, return_dict=True).to(model.device)
with torch.inference_mode():
    audio = model.generate(**inputs, output_audio=True)
outfile = f'dhivehi_voice_{speaker_id}.wav'
processor.save_audio(audio, outfile)
from IPython.display import display, Audio
display(Audio(filename=outfile))
print('Saved:', outfile)


In [ ]:
from google.colab import files
files.download(outfile)
